# CNN for CIFAR10

In [101]:
import torch
import torch.nn.functional as F
from torch import nn

In [102]:
import math
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib.pyplot import imread
import scipy
from PIL import Image
import pandas as pd

from typing import Sequence

## Model

### Resnet

In [103]:
class relu2(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return F.relu(x) + F.relu(x) * F.relu(x)

class BasicResidualBlock(nn.Module):
    """
        Start with basic block first (for resnet18, resnet34).

        input >  3×3 Conv (stride) > BN > ReLU > 3×3 Conv (1) > BN > Add > ReLU (same size accross the dataflow)

        *In this case, the cifar10 dataset is not complicated enough for the bottleneck block to make any differences. > use basic block for now.*
    """

    expansion = 1

    def __init__(self, 
                 input_dim: int, # number of channels
                 planes: int, # internal width per stage in net
                 stride: int = 1, # initial stride    
                 downsample: nn.Module = None, # block of shortcut
        ):
        super(BasicResidualBlock, self).__init__()

        output_channels = planes * self.expansion

        self.residual = nn.Sequential(
            nn.Conv2d(input_dim, planes, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(planes),
            # nn.ReLU(),
            relu2(),

            nn.Conv2d(planes, output_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(output_channels),
        )

        if downsample:
            self.downsample = downsample
        else:
            self.downsample = nn.Identity()
    
    def forward(self, x):
        identity = self.downsample(x)
        residual = self.residual(x)
        return identity + F.relu(residual)

class resnet(nn.Module):
    def __init__(self,
        Block,
        layers: Sequence[int],
        num_classes: int = 10,
        input_channels: int = 3):

        super(resnet, self).__init__()

        self.in_channels = 64
        # Initial feature extraction
        self.conv1 = nn.Sequential(
            nn.Conv2d(input_channels, 64, kernel_size=7, stride=2, padding=3),
            # nn.Conv2d(input_channels, 64, kernel_size=3, stride=1, padding=1),
            # nn.BatchNorm2d(64),
            nn.GroupNorm(4,64, bias=False),
            # nn.ReLU(),
            relu2(),
        )

        # Maxpool
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # self.maxpool = nn.Identity()

        # Layers
        self.big_layers = nn.Sequential(
            self._make_layer(Block=Block, planes=64, number_of_blocks=layers[0], stride=1),
            self._make_layer(Block=Block, planes=128, number_of_blocks=layers[1], stride=2),
            self._make_layer(Block=Block, planes=256, number_of_blocks=layers[2], stride=2),
            self._make_layer(Block=Block, planes=512, number_of_blocks=layers[3], stride=2),   
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))

        self.classification_head = nn.Sequential(
            nn.Linear(512, 256),
            nn.GELU(),

            nn.Linear(256, 128),
            nn.GELU(),

            nn.Linear(128, 64),
            nn.GELU(),
            
            nn.Linear(64, num_classes),
        )

        # self.classification_head = nn.Sequential(
        #     nn.Linear(512, num_classes),
        #     # nn.GELU(),
        #     # nn.Linear(...),
        # )

    def _make_layer(self,
        Block: BasicResidualBlock,
        planes: int,
        number_of_blocks: int,
        stride: int = 1):

        output_channels = planes * Block.expansion

        layers = []
        downsample = None

        if stride != 1 or self.in_channels != output_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, output_channels, kernel_size=1, stride=stride, padding=0),
                nn.GroupNorm(4,output_channels, bias=False)
            )

        layers.append(Block(input_dim=self.in_channels, 
                         planes=planes, 
                         downsample=downsample,
                         stride=stride))

        self.in_channels = output_channels

        for i in range(1, number_of_blocks):
            layers.append(Block(self.in_channels, planes=planes))

        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.big_layers(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)  # Flatten the tensor
        return self.classification_head(x)

 

In [104]:
model = resnet(BasicResidualBlock, [2, 2, 2, 2])
x = torch.randn(4, 3, 32, 32)
output = model(x)

print(output.shape)
# torch.Size([4, 10])

torch.Size([4, 10])


### Basic CNN

In [105]:
# # input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
# class CNN(nn.Module):
#     """
#     Parameters:
#            * in_channels: Number of channels in the input image (for grayscale images, 1)
#            * num_classes: Number of classes to predict.
           
#     """
#     def __init__(self, input_dim: int, output_dim: int):
#         super(CNN, self).__init__()

#         self.model = nn.Sequential(
#             nn.Conv2d(input_dim, 16, kernel_size=3, stride=1, padding=1), # Conv 1
#             # nn.BatchNorm2d(16),
#             nn.GELU(),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1), # Conv 2
#             # nn.BatchNorm2d(32),
#             nn.GELU(),
#             nn.Dropout(0.2),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1), # Conv 3
#             # nn.BatchNorm2d(64),
#             nn.GELU(),
#             nn.Dropout(0.3),

#             nn.MaxPool2d(kernel_size=2, stride=2), # Max polling

#             nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1), # Conv 4 
#             # nn.BatchNorm2d(64),
#             nn.GELU(),

#         )
#         self.fc1 = nn.Linear(64 * 4 * 4, 128)
#         self.fc2 = nn.Linear(128, output_dim) # Output layer, output_dim = number of classes output

#     def forward(self, x):
#         """
#             Model flow: input -> conv -> relu -> pool -> conv -> relu -> pool -> fc -> softmax (output)
#         """
#         x = self.model(x)
#         x = x.reshape(x.shape[0], -1)  # Flatten the tensor
#         # print(x.shape)
#         x = F.gelu(self.fc1(x)) # Apply fully connected layer 
#         x = self.fc2(x) # Output layer
#         return x

## Data Loading

In [106]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [107]:
BATCH_SIZE = 64

In [108]:
# Calculate z-score for normalization

# temp_train_dataset = datasets.CIFAR10(
#     root='dataset/',
#     train=True,
#     transform=transforms.ToTensor(),
#     download=True
# )

# temp_train_loader = DataLoader(
#     dataset=temp_train_dataset,
#     batch_size=BATCH_SIZE,
#     shuffle=False,
#     num_workers=2
# )

# mean = 0.
# std = 0.
# for images, _ in temp_train_loader:
#     batch_samples = images.size(0) # batch size (the last batch can have smaller size!)
#     images = images.view(batch_samples, images.size(1), -1)
#     mean += images.mean(2).sum(0)
#     std += images.std(2).sum(0)

# mean /= len(temp_train_loader.dataset)
# std /= len(temp_train_loader.dataset)

# mean = mean.tolist()
# std = std.tolist()

# print("Calculated mean:", mean)
# print("Calculated std:", std)

In [109]:
train_dataset = datasets.CIFAR10(root='dataset/', train=True, transform=transforms.Compose(
            [
                transforms.ToTensor(),
                transforms.RandomCrop(32, padding=4),
                transforms.RandomHorizontalFlip(),
                # transforms.Lambda(lambda t: t / 255.0), 
                # transforms.Normalize(mean=mean, std=std),
                transforms.Lambda(lambda x: 2 * x - 1),
                transforms.RandomVerticalFlip(),
                # transforms.GaussianBlur(3, 0.03),
            ]
        ), download=True)
test_dataset = datasets.CIFAR10(root='dataset/', train=False, transform=transforms.Compose(
            [
                transforms.ToTensor(),
                # transforms.Lambda(lambda t: t / 255.0), 
                # transforms.Normalize(mean=mean, std=std),
                transforms.Lambda(lambda x: 2 * x - 1)
            ]
        ), download=True)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

In [110]:
train_dataset[0][0].shape

torch.Size([3, 32, 32])

In [111]:
len(set(train_dataset.targets))

10

## Training

In [112]:
input_dim = train_dataset[0][0].shape[0]
output_dim = len(set(train_dataset.targets))

### Resnet

In [113]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = resnet(Block=BasicResidualBlock, 
             layers=[3,4,6,3],
             num_classes=output_dim,
             input_channels=input_dim).to(device)
# net = torch.compile(net, fullgraph=True)

In [114]:
net.eval

<bound method Module.eval of resnet(
  (conv1): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
    (1): GroupNorm(4, 64, eps=1e-05, affine=True, bias=False)
    (2): relu2()
  )
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (big_layers): Sequential(
    (0): Sequential(
      (0): BasicResidualBlock(
        (residual): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): relu2()
          (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
          (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
        (downsample): Identity()
      )
      (1): BasicResidualBlock(
        (residual): Sequential(
          (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding

In [115]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter("runs/resnet_cifar10")

In [116]:
epochs = 100
criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
optimizer = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
# optimizer = torch.optim.SGD(
#     model.parameters(),
#     lr=0.1,
#     momentum=0.9,
#     weight_decay=1e-2,
#     nesterov=True,
# )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

### Basic CNN

In [117]:
# from torch.utils.tensorboard import SummaryWriter
# writer = SummaryWriter("runs/cnn_cifar10")

In [118]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(device)
# model = CNN(input_dim=input_dim, output_dim=output_dim).to(device)

In [119]:
# epochs = 30
# criterion = nn.CrossEntropyLoss() # Apply softmax as an activation func for the output layer
# optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
# # scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.5)
# scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

In [120]:
# for param_tensor in model.state_dict():
#     print(param_tensor, "\t", model.state_dict()[param_tensor].size())

In [121]:
# model.eval

### Accuracy

In [122]:
def check_accuracy(loader, model):
    num_correct = 0
    num_samples = 0

    model.eval()

    with torch.inference_mode():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            scores = model(x)
            _, predictions = scores.max(1)

            num_correct += (predictions == y).sum().item()
            num_samples += predictions.size(0)

    return num_correct / num_samples

### Run

In [123]:
def run(model, optimizer, scheduler):
    step = 0

    for epoch in range(epochs):
        running_loss = 0.0
        model.train()

        for batch_idx, (data, targets) in enumerate(train_loader):
            data = data.to(device)
            targets = targets.to(device)

            # Add Gaussian noise using the same device and shape as data
            data = data + torch.randn_like(data) * 0.03

            # Forward propagation
            scores = model(data)
            loss = criterion(scores, targets)

            # Backward propagation
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            running_loss += loss.item()

            writer.add_scalar("Training Loss / Batch", loss.item(), step,)

            step += 1

        avg_epoch_loss = running_loss / len(train_loader)

        train_acc = check_accuracy(train_loader, model)
        test_acc = check_accuracy(test_loader, model)

        writer.add_scalar("Training Loss / Epoch", avg_epoch_loss, epoch,)
        writer.add_scalar("Training Accuracy", train_acc, epoch,)
        writer.add_scalar("Test Accuracy", test_acc, epoch,)

        # Step the scheduler once per epoch
        scheduler.step()

        # Log the learning rate used for the next epoch
        current_lr = optimizer.param_groups[0]["lr"]
        writer.add_scalar("Learning Rate", current_lr, epoch)

        print(
            f"Epoch [{epoch + 1}/{epochs}], "
            f"Loss: {avg_epoch_loss:.4f}, "
            f"Train Acc: {train_acc:.4f}, "
            f"Test Acc: {test_acc:.4f}, "
            f"LR: {current_lr:.6f}"
        )

    writer.close()

#### Run - resnet

In [124]:
run(net, optimizer=optimizer, scheduler=scheduler)

Epoch [1/100], Loss: 1.8510, Train Acc: 0.3257, Test Acc: 0.3439, LR: 0.001000
Epoch [2/100], Loss: 1.6157, Train Acc: 0.4158, Test Acc: 0.4137, LR: 0.000999
Epoch [3/100], Loss: 1.4427, Train Acc: 0.4884, Test Acc: 0.4850, LR: 0.000998
Epoch [4/100], Loss: 1.3196, Train Acc: 0.5371, Test Acc: 0.5357, LR: 0.000996
Epoch [5/100], Loss: 1.2351, Train Acc: 0.5756, Test Acc: 0.5787, LR: 0.000994
Epoch [6/100], Loss: 1.1613, Train Acc: 0.5797, Test Acc: 0.5885, LR: 0.000991
Epoch [7/100], Loss: 1.1095, Train Acc: 0.6037, Test Acc: 0.5969, LR: 0.000988
Epoch [8/100], Loss: 1.0613, Train Acc: 0.6314, Test Acc: 0.6361, LR: 0.000984
Epoch [9/100], Loss: 1.0221, Train Acc: 0.6589, Test Acc: 0.6388, LR: 0.000980
Epoch [10/100], Loss: 0.9855, Train Acc: 0.6332, Test Acc: 0.6091, LR: 0.000976
Epoch [11/100], Loss: 0.9480, Train Acc: 0.6662, Test Acc: 0.6387, LR: 0.000970
Epoch [12/100], Loss: 0.9205, Train Acc: 0.6486, Test Acc: 0.6231, LR: 0.000965
Epoch [13/100], Loss: 0.8930, Train Acc: 0.6674, 

#### Run - CNN

In [125]:
# run(model)

## Benchmark

### Goal

CIFAR_10
* SOTA training acc = 94-96%
* Good test acc: 85%+
* Gap = around 5%–10% - Not use augmentation. 
* Gap = 2%–5% - With augmentation, BatchNorm, dropout, and weight decay.

### Fine-tunning Result

#### v1.0
Architecture + hyperparameters:

    * Flow = input -> conv(16,3,1,1) -> gelu -> maxpool(2,2) -> conv(32,3,1,1) -> gelu -> dropout(0.2) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> dropout(0.3) -> maxpool(2,2) -> conv(64,3,1,1) -> gelu -> Flatten shape=(64, 64 * 4 * 4) -> Linear(64 * 4 * 4, 10) -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = 1e-3

    * # epoch = 20, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: 0.8922, Test Acc: 0.7565

Resnet34

Architecture + hyperparameters:

    * Flow (paper resnet-34 based) = input -> conv(64,7,2,3) -> batchnorm(64) -> relu -> maxpool(3,2,1) -> {residual block 3 x [conv(64,3,1,1) -> batchnorm(64) -> relu -> conv(64,3,1,1) -> batchnorm(64) -> relu] -> residual block 4 x [conv(128,3,2,1) -> batchnorm(128) -> relu -> conv(128,3,1,1) -> batchnorm(128) -> relu] -> residual block 6 x [conv(256,3,2,1) -> batchnorm(256) -> relu -> conv(256,3,1,1) -> batchnorm(256) -> relu] -> residual block 3 x [conv(512,3,2,1) -> batchnorm(512) -> relu -> conv(512,3,1,1) -> batchnorm(512) -> relu]} -> AdaptiveAvgPool2d(1,1) -> Flatten -> [Linear(512, 256) -> gelu -> Linear(256, 128) -> gelu -> Linear(128, 64) -> gelu -> Linear(64, 10)] -> (CrossEntropyLoss)

    * Optimiser = AdamW
    
    * Learning rate = lr = 1e-3, weight_decay=1e-2

    * scheduler => 1e - 6

    * # epoch = 100, batch_size = 64

    * Transform: normalising to [-1, 1]

    * Noise: 0.03

Result:
    
    Train Acc: , Test Acc: 